In [ ]:
import json
import ssl
import urllib.parse
import urllib.request

import certifi

params = urllib.parse.urlencode(
    {
        "start": "1",
        "limit": "5000",
        "convert": "USD",
    }
)

request = urllib.request.Request(
    f"https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest?{params}",
    headers={
        "Accept": "application/json",
        "X-CMC_PRO_API_KEY": "YOUR_COINMARKETCAP_API_KEY",
    },
)

context = ssl.create_default_context(cafile=certifi.where())

with urllib.request.urlopen(request, context=context) as response:
    data = json.load(response)

print(len(data['data']))

#NOTE:
# I had to go in and put "jupyter notebook --NotebookApp.iopub_data_rate_limit=1e10"
# Into the Anaconda Prompt to change this to allow to pull data

In [ ]:
type(data)

In [ ]:
import pandas as pd
#This allows you to see all the columns, not just like 15
pd.set_option('display.max_columns', None)

In [ ]:
#This normalizes the data and makes it all pretty in a dataframe
df = pd.json_normalize(data['data'])
df['timestamp'] = pd.to_datetime('now')

df

In [ ]:

import pandas as pd
import os
from time import sleep

# Create empty dataframe
df = pd.DataFrame()

def api_runner():
    global df

    params = urllib.parse.urlencode(
        {
            "start": "1",
            "limit": "15",
            "convert": "USD",
        }
    )

    request = urllib.request.Request(
        f"https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest?{params}",
        headers={
            "Accept": "application/json",
            "X-CMC_PRO_API_KEY": "YOUR_COINMARKETCAP_API_KEY"
        },
    )

    context = ssl.create_default_context(cafile=certifi.where())

    with urllib.request.urlopen(request, context=context) as response:
        data = json.load(response)

    # Convert JSON to DataFrame
    df2 = pd.json_normalize(data['data'])

    # Add timestamp
    df2['timestamp'] = pd.to_datetime('now')

    # Append to main DataFrame
    df = pd.concat([df, df2], ignore_index=True)

    # File path
    file_path = r'C:\Users\User\Desktop\New folder\API.csv'

    # Save data
    if not os.path.isfile(file_path):
        df2.to_csv(file_path, index=False)
    else:
        df2.to_csv(file_path, mode='a', header=False, index=False)


# Run API every minute
for i in range(333):
    api_runner()
    print(f'API Runner Completed - Iteration {i+1}')
    sleep(60)

print("Finished!")

In [ ]:
df72 = pd.read_csv(r'C:\Users\User\Desktop\New folder\API.csv')
df72

In [ ]:
pd.set_option('display.float_format', lambda x: '%.5f' % x)
df


In [ ]:
df3 = df.groupby('name', sort=False)[['quote.USD.percent_change_1h','quote.USD.percent_change_24h','quote.USD.percent_change_7d','quote.USD.percent_change_30d','quote.USD.percent_change_60d','quote.USD.percent_change_90d']].mean()
df3

In [ ]:
df4 = df3.stack()
df4

In [ ]:
type(df4)

In [ ]:
df5 = df4.to_frame('values')
df5

In [ ]:
df5.count()

In [ ]:
index = pd.Index(range(90))

df6 = df5.reset_index()
df6

In [ ]:
df7 = df6.rename(columns = {'level_1':'percent_change'})
df7

In [ ]:
df7['percent_change'] = df7['percent_change'].replace(['quote.USD.percent_change_1h','quote.USD.percent_change_24h','quote.USD.percent_change_7d','quote.USD.percent_change_30d','quote.USD.percent_change_60d','quote.USD.percent_change_90d'],['1h','24h','7d','30d','60d','90d'])
df7

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sns.catplot(x = 'percent_change', y = 'values', hue = 'name', data = df7, kind = 'point')

In [ ]:
df10 = df[['name','quote.USD.price','timestamp']]
df10 = df10.query("name == 'Bitcoin'")
df10